In [10]:
import os
import json
import asyncio
import time
from mistralai import Mistral
from tavily import TavilyClient
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import List


# Chargement forcé de ta configuration
load_dotenv("cle.env", override=True)

# Initialisation des clients
client = Mistral(api_key=os.getenv("MISTRAL_API_KEY"))
tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

class AnalysisSchema(BaseModel):
    summary: str
    sources: List[SourceSchema]
class ClaimSchema(BaseModel):
    text: str
class JugeSchema(BaseModel):
    claim: ClaimSchema
    analysis: AnalysisSchema
    overall_verdict: str = Field(description="Valeurs : accurate, inaccurate, exaggerated, misleading, partially_accurate")
print("✅ Moteurs Mistral et Tavily prêts.")

✅ Moteurs Mistral et Tavily prêts.


In [89]:
# La liste qui sauve ta démo
FR_ELITE_DOMAINS = [
    "gouv.fr",             # TOUS les ministères et services de l'État
    "insee.fr",            # La base pour les stats
    "vie-publique.fr",     # Synthèses administratives
    "ccomptes.fr",         # Cour des comptes
    "senat.fr",            # Rapports législatifs
    "assemblee-nationale.fr",
    "ameli.fr",            # Sécu / Santé
    "lemonde.fr",          # Presse de référence
    "lesechos.fr",         # Économie
    "radiofrance.fr",      # France Info / Inter
    "francetvinfo.fr",     # Service public
    "service-public.fr",   # Infos administratives
    "rte-france.com"       # Nucléaire / Électricité
] 
class RouteurSchema(BaseModel):
    affirmation_propre: str
    run_stats: bool
    run_rhetorique: bool
    run_coherence_personnelle: bool
    run_contexte: bool
    a_verifier: bool

SOCIAL_BLACKLIST = ["tiktok.com", "facebook.com", "instagram.com", "x.com", "twitter.com", "reddit.com", "quora.com", "pinterest.com"]

async def agent_statistique(data):
    # On force la recherche sur les chiffres et les domaines officiels
    query = f"chiffre officiel précis {data['affirmation']} (site:gouv.fr OR site:insee.fr OR site:rte-france.com)"
    
    try:
        search = tavily_client.search(query=query, search_depth="advanced", max_results=5)
        sources = "\n\n".join([f"SOURCE {i+1} ({r['url']}): {r['content']}" for i, r in enumerate(search['results'])])
    except: sources = "Pas de sources."

    prompt = f"""Tu es un extracteur de données. Ton seul but est de remplir ces cases.
    AFFIRMATION : {data['affirmation']}
    SOURCES : {sources}

    CONSIGNES :
    1. Trouve le chiffre cité par le sujet. S'il dit "aucun", mets 0.
    2. Trouve le chiffre réel dans les sources. 
    3. Si tu vois une évolution (ex: -20%), utilise ce pourcentage comme chiffre_reel.
    4. Pour 'nom_source', utilise uniquement l'acronyme (INSEE, RTE, etc.).

    INTERDICTION : Ne fais aucun calcul. Ne convertis rien. Prends les nombres tels quels.
    """
    # Utilise ici ton appel API habituel avec response_format JSON
    
    res = await client.chat.complete_async(
        model="mistral-small-latest",
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )
    return json.loads(res.choices[0].message.content)

    
# --- AGENT 2 : RHÉTORIQUE ---
async def agent_rhetorique(data):
    print(f"🧠 [Agent Rhétorique] Analyse logique...")
    prompt = f"""Analyse : Question posée : '{data.get('question_posee', '')}' | Réponse : '{data.get('affirmation', '')}'
    RÈGLES STRICTES :
    1. Si la personne répond à la question (ou s'il n'y avait pas de question) : Laisse "explication" VIDE "".
    2. Si la personne esquive : Explique en une phrase qu'elle ne répond pas à la question posée.
    
    JSON: {{"agent": "rhetorique", "explication": "..."}}"""
    
    res = await client.chat.complete_async(model="mistral-small-latest", messages=[{"role": "user", "content": prompt}], response_format={"type": "json_object"})
    return json.loads(res.choices[0].message.content)

# --- AGENT 3 : COHÉRENCE PERSONNELLE ---
async def agent_coherence_personnelle(data):
    print(f"🕵️ [Agent Cohérence] Accès réseaux sociaux pour {data['personne']}...")
    try:
        search = tavily_client.search(
            query=f"déclaration {data['personne']} {data['affirmation']} 2025 2026", search_depth="advanced", max_results=3
        )
        sources = "\n".join([f"Source: {r['url']}\nContenu: {r['content']}" for r in search.get('results', [])])
    except: sources = "Aucune archive."

    prompt = f"""Vérifie si {data['personne']} se contredit sur : '{data['affirmation']}'. Sources : {sources}.
    RÈGLES STRICTES :
    1. Si la personne est cohérente : Laisse "explication" VIDE "".
    2. Si incohérente : Cite brièvement les propos incohérents.
    
    JSON: {{"agent": "coherence", "explication": "..."}}"""
    
    res = await client.chat.complete_async(model="mistral-small-latest", messages=[{"role": "user", "content": prompt}], response_format={"type": "json_object"})
    return json.loads(res.choices[0].message.content)

# --- AGENT 4 : CONTEXTE (MODE INVESTIGATION LIBRE) ---
async def agent_contexte(data):
    print(f"📚 [Agent Contexte] Analyse factuelle détaillée...")
    
    query_complete = f"{data['affirmation']} contexte faits historiques France 2025 2026"

    try:
        search = tavily_client.search(
            query=query_complete, 
            search_depth="basic", 
            max_results=3, 
            exclude_domains=SOCIAL_BLACKLIST
        )
        sources = "\n".join([f"Source: {r['url']}\nContenu: {r['content']}" for r in search.get('results', [])])
    except: 
        sources = "Aucun contexte."

    prompt = f"""Analyse le contexte de : '{data['affirmation']}'. Sources : {sources}.
    ⚠️ ATTENTION CRITIQUE : Cette affirmation peut être un MENSONGE TOTAL. Ne la prends pas pour une vérité absolue.
    
    RÈGLES STRICTES :
    1. analyse_detaillee : Fais une analyse approfondie (environ 5 à 7 phrases) pour expliquer le contexte RÉEL. Si les sources contredisent l'affirmation, explique la vraie situation.
    2. nom_source : Nom de la source principale trouvée.
    3. url_source : L'URL de cette source.
    
    JSON: {{"agent": "contexte", "analyse_detaillee": "...", "nom_source": "...", "url_source": "..."}}"""
    
    res = await client.chat.complete_async(
        model="mistral-small-latest", 
        messages=[{"role": "user", "content": prompt}], 
        response_format={"type": "json_object"}
    )
    return json.loads(res.choices[0].message.content)


In [94]:
# --- ROUTEUR (CORRIGÉ POUR DÉTECTER LES MOTS QUANTITATIFS) ---
async def agent_routeur(data):
    prompt = f"""Tu es le cerveau analytique d'un système de fact-checking en direct.
    ENTRÉE BRUTE : '{data.get('affirmation', '')}'

    MISSION 1 : LE TAMIS FACTUEL
    Analyse chaque proposition de la phrase. Il s'agit d'une transcription donc il faut être vigilant.
    - TRADUIS correctement la phrase : vérifie que les mots soient cohérents en eux, sinon corrige la pour qu'elle soit cohérente (par exemple : J'ai un cuit de 130 = j'ai un QI de 130)
    - GÈRE LES AUTO-CORRECTIONS : Si la personne se reprend (ex: "20%... enfin 65%"), SUPPRIME l'erreur. Ne garde QUE l'affirmation finale.
    - ÉLIMINE le "bruit" : goûts personnels ("J'aime les chips"), opinions subjectives ("La dignité est importante"), ou rhétorique vague.
    - GARDE uniquement le "signal" : affirmations avec des chiffres, des dates, des noms propres, ou des comparaisons (plus de, moins de, record).

    MISSION 2 : NETTOYAGE ET SYNTHÈSE
    Le texte contient une seule phrase.
    - Si plusieurs faits sont présents (ex: QI et Population), combine-les en une seule "affirmation_propre".
    - Si aucun fait n'est détecté, renvoie une chaîne vide pour "affirmation_propre".

    MISSION 4 : ROUTAGE (RÈGLES ULTRA STRICTES)
    Sur la base de cette "affirmation_propre", active les experts (true/false) :
    - 'run_stats' : TRUE pour les chiffres, pourcentages, budgets, économie, OU les affirmations quantitatives absolues (ex: "aucun", "plus de", "tout le monde", "zéro").
    - 'run_rhetorique' : TRUE si une question a été posée ou si le discours esquive.
    - 'run_coherence_personnelle' : TRUE si la personne parle de son passé ou de ses propres déclarations.
    - 'run_contexte' : TRUE pour des évènements précis (JO, manifestations, lois) ou de grandes affirmations factuelles vérifiables (ex: "La France est le pays qui taxe le plus"). NE PAS ACTIVER pour des données purement chiffrées.

    Renvoie UNIQUEMENT un JSON strict :
    {{
      "affirmation_propre": "La phrase nettoyée",
      "run_stats": bool,
      "run_rhetorique": bool,
      "run_coherence_personnelle": bool,
      "run_contexte": bool
    }}
    """
    res = await client.chat.parse_async(
        model="mistral-small-latest", 
        messages=[{"role": "user", "content": prompt}], 
        response_format=RouteurSchema
    )
    return res.choices[0].message.parsed.model_dump()

# --- EXÉCUTEUR PARALLÈLE ---
async def executer_analyse_parallele(data):
    # 1. Le routeur nettoie (et gère les "enfin", "pardon", "je veux dire")
    routage = await agent_routeur(data)
    affirmation_propre = routage.get('affirmation_propre', data['affirmation'])
    
    # Debug pour toi (demandé)
    print(f"📡 ROUTAGE : [Stats: {routage['run_stats']}] | [Contexte: {routage['run_contexte']}] | [Rhétorique: {routage['run_rhetorique']}]")
    print(f"✨ TEXTE NETTOYÉ : '{affirmation_propre}'")
    
    if not routage.get("a_verifier") or not affirmation_propre:
        return affirmation_propre, [] 

    data_propre = data.copy()
    data_propre['affirmation'] = affirmation_propre 
    
    taches = []
    if routage.get("run_stats"): taches.append(agent_statistique(data_propre))
    if routage.get("run_rhetorique"): taches.append(agent_rhetorique(data_propre))
    if routage.get("run_coherence_personnelle"): taches.append(agent_coherence_personnelle(data_propre))
    if routage.get("run_contexte"): taches.append(agent_contexte(data_propre))
        
    rapports = await asyncio.gather(*taches)
    return affirmation_propre, rapports # 🔥 On renvoie les deux !

async def agent_editeur(affirmation, rapports):
    # Extraction des preuves pour le prompt
    preuves_stats = ""
    for r in rapports:
        if r.get('agent') == 'statistique':
            preuves_stats += f"DONNÉE RÉELLE : {r.get('chiffre_reel')} {r.get('unite')} provenant de {r.get('nom_source')}\n"

    prompt = f"""Tu es un robot de data-visualisation pour une chaîne d'info.
    AFFIRMATION : "{affirmation}"
    PREUVES TROUVÉES : {preuves_stats}

    RÈGLES D'OR POUR LE 'summary' :
    1. Si l'affirmation est fausse, le summary DOIT être : "FAUX : [Chiffre Réel] [Unité] (Source : [Nom de la Source])".
    2. INTERDICTION de faire des phrases comme "L'affirmation est fausse car...".
    3. Si l'expert n'a pas trouvé de chiffre, utilise le contexte pour démentir brièvement, mais privilégie toujours la stat.
    4. Sois ultra-précis. Pas de politesse.
    """

    res = await client.chat.parse_async(
        model="mistral-small-latest", 
        messages=[{"role": "user", "content": prompt}], 
        response_format=JugeSchema
    )
    return res.choices[0].message.parsed.model_dump()

In [95]:
# --- TEST DU NOUVEAU WORKFLOW ---
async def tester_pipeline_tv():
    # Simulation des données envoyées par ton outil (Voxtral)
    contexte_historique = "Je pense qu'il y a 30% de SDF à Paris. Il y a 90% de base en France. La dette de la sécurité sociale est de 30 milliards."
    
    data_instant_t = {
        "personne": "Gabriel Attal",
        "question_posee": "",
        "affirmation": "La dette de la sécurité sociale est de 30 milliards. J'aime le chocolat et les beignets"
    }

    start_time = time.perf_counter()
    
    # 1. Analyse brute par les agents
    
   # 🔥 ICI : On déballe les deux résultats renvoyés par l'exécuteur
    affirmation_nettoyee, rapports_bruts = await executer_analyse_parallele(data_instant_t)
    
    # 2. On donne la phrase NETTOYÉE et les RAPPORTS au Juge
    bandeau_tv = await agent_editeur(
        contexte_precedent=contexte_historique, 
        affirmation_actuelle=affirmation_nettoyee, # On utilise la version propre
        rapports_agents=rapports_bruts
    )
    
    end_time = time.perf_counter()
    
    print(f"\n⏱️ Temps total du Pipeline : {end_time - start_time:.2f}s")
    print("\n--- BANDEAU OBS FINAL ---")
    print(json.dumps(bandeau_tv, indent=2, ensure_ascii=False))

# Décommente cette ligne pour tester le nouveau système !
await tester_pipeline_tv()


📡 ROUTAGE : [Stats: True] | [Contexte: False] | [Rhétorique: False]
✨ TEXTE NETTOYÉ : 'La dette de la sécurité sociale est de 30 milliards'


TypeError: agent_editeur() got an unexpected keyword argument 'contexte_precedent'

In [92]:
dataset_pipeline_expert = [
    # --- 5 EXEMPLES AVEC QUESTIONS (Q&A) ---
    {
        "p": "Gabriel Attal", "q": "Que faites-vous pour les sans-abris ?",
        "ctx": "La dignité humaine est au cœur de notre action. Nous avons ouvert 200 000 places d'hébergement d'urgence. L'État mobilise des moyens sans précédent cet hiver. Les préfets ont reçu des instructions claires pour ne laisser personne dehors. Nous avons investi 2 milliards d'euros dans le plan Logement d'Abord. Les associations sont nos partenaires quotidiens. La solidarité nationale doit jouer à plein. Nous ne laissons personne sur le côté. Le combat contre la pauvreté est une priorité.",
        "a": "Grâce à notre action, il n'y a plus aucun SDF en France aujourd'hui."
    },
    {
        "p": "Jordan Bardella", "q": "Quelle est votre position sur l'énergie ?",
        "ctx": "La France doit retrouver sa puissance électrique. Le marché européen nous impose des prix délirants qui ruinent les familles. Nous avons un parc nucléaire exceptionnel mais il a été saboté par l'idéologie. Il faut arrêter de fermer des centrales. La souveraineté énergétique est la base de l'indépendance nationale. Nos entreprises délocalisent à cause du coût de l'énergie. Le gaz russe a été remplacé par du gaz de schiste américain plus cher. C'est un non-sens écologique et économique.",
        "a": "Aujourd'hui, le nucléaire ne représente plus que 20% de notre électricité... enfin, 65%."
    },
    {
        "p": "Marine Tondelier", "q": "Faut-il interdire les pesticides ?",
        "ctx": "La santé des Français n'est pas négociable. L'eau que nous buvons est polluée par des molécules chimiques. Les agriculteurs sont les premières victimes de ces produits. On observe une chute dramatique de la biodiversité dans nos campagnes. Les abeilles disparaissent à un rythme alarmant. Le gouvernement recule sans cesse face aux lobbies industriels. Il faut accompagner la transition au lieu de la punir. L'agroécologie est la seule voie de survie pour nos sols.",
        "a": "L'utilisation du glyphosate a augmenté de 50% en France cette année."
    },
    {
        "p": "Bruno Le Maire", "q": "Où en sont les impôts ?",
        "ctx": "Nous avons fait le choix de la politique de l'offre. Baisser les impôts des entreprises permet de créer des emplois. Nous avons supprimé la taxe d'habitation pour tous les Français. La pression fiscale était trop forte dans ce pays. Nous voulons récompenser le travail et l'effort. Les prélèvements obligatoires ont commencé à refluer. C'est une trajectoire de long terme que nous tenons. La croissance revient grâce à cette stabilité fiscale.",
        "a": "Nous avons réussi à diviser par deux la dette de la France en cinq ans."
    },
    {
        "p": "Sandrine Rousseau", "q": "Pourquoi cibler les jets privés ?",
        "ctx": "On demande des efforts de sobriété aux Français modestes. On leur dit de baisser le chauffage à 19 degrés. Mais les ultra-riches continuent de brûler la planète en toute impunité. Un trajet en jet émet plus de CO2 qu'un Français moyen en un an. C'est une injustice climatique insupportable. Le gouvernement refuse de réguler ce secteur. L'écologie ne peut pas être punitive pour les pauvres et permissive pour les riches.",
        "a": "Mais la vraie question, c'est de savoir si le gouvernement va taxer les superprofits de Total !"
    },
    # --- 10 EXEMPLES DISCOURS BRUTS (SANS QUESTIONS) ---
    {
        "p": "Emmanuel Macron", "q": "",
        "ctx": "Nous vivons une période de grandes transformations mondiales. La France doit être aux avant-postes de la technologie. Nous investissons massivement dans l'IA et le quantique. L'école est le berceau de cette ambition future. Il faut réarmer notre éducation nationale. Le niveau en mathématiques doit devenir une priorité dès le CP. Nous avons déjà commencé à recruter davantage de professeurs. La formation continue sera le pilier de notre réforme.",
        "a": "Nous avons recruté 100 000 policiers supplémentaires cette année."
    },
    {
        "p": "Jean-Luc Mélenchon", "q": "",
        "ctx": "Le peuple a faim pendant que les riches se gavent. Les prix à la consommation étranglent les familles. On nous parle de croissance, mais la croissance de quoi ? De la misère ! Nous proposons le blocage des prix des produits de première nécessité. Il faut augmenter le SMIC immédiatement. La vie chère n'est pas une fatalité, c'est un choix politique. Les banques centrales ne font qu'aggraver la situation en montant les taux.",
        "a": "Le SMIC en France est actuellement à 2500 euros net."
    },
    {
        "p": "Marine Le Pen", "q": "",
        "ctx": "La sécurité est la première des libertés. Nos villes et nos villages sont livrés à une délinquance de plus en plus barbare. Le laxisme judiciaire encourage les récidivistes. Il faut rétablir l'autorité partout. Les forces de l'ordre doivent être soutenues par l'État. Nous avons besoin de 85 000 places de prison supplémentaires. Le lien entre immigration et insécurité n'est plus à démontrer pour personne. Les Français veulent vivre en paix.",
        "a": "Le taux de cambriolages a baissé de 40% en un an sous ce gouvernement."
    },
    {
        "p": "Rachida Dati", "q": "",
        "ctx": "La culture est le ciment de notre Nation. Elle ne doit pas être réservée aux élites parisiennes. Chaque village doit avoir accès à une programmation de qualité. Nous soutenons le patrimoine religieux et rural. C'est l'âme de nos territoires. Le pass culture est un outil formidable de démocratisation. Nous allons renforcer les moyens des bibliothèques de quartier. L'art doit entrer dans les écoles dès le plus jeune âge.",
        "a": "Le budget de la culture est le premier poste de dépense de l'État."
    },
    {
        "p": "Olivier Faure", "q": "",
        "ctx": "La réforme des retraites est une blessure sociale qui ne se refermera pas. Travailler jusqu'à 64 ans est une injustice pour ceux qui ont des carrières longues. Les femmes sont les grandes perdantes de cette loi. Nous proposons le retour à la retraite à 60 ans. La pénibilité doit être réellement prise en compte. On ne peut pas traiter un ouvrier comme un cadre. Le financement est possible en taxant les revenus du capital.",
        "a": "La réforme des retraites a été votée à l'unanimité par le Parlement."
    },
    {
        "p": "Eric Ciotti", "q": "",
        "ctx": "La France subit un dérapage budgétaire incontrôlé. Le déficit public atteint des sommets alarmants. Le gouvernement dépense l'argent qu'il n'a pas. Nos enfants hériteront d'une dette colossale. Il faut sabrer dans la dépense publique inutile. L'État doit se concentrer sur ses missions régaliennes. Nous sommes les champions du monde des prélèvements obligatoires. Il faut libérer les entreprises de ce poids mort administratif.",
        "a": "Le déficit public de la France est tombé à 1% du PIB en 2025."
    },
    {
        "p": "Manuel Bompard", "q": "",
        "ctx": "La Ve République est une monarchie présidentielle. Un seul homme décide de tout pour 67 millions de citoyens. L'article 49.3 est devenu l'outil de la brutalité politique. Nous devons passer à une VIe République parlementaire. Le référendum d'initiative citoyenne doit être instauré. Il faut redonner du pouvoir aux élus locaux. La démocratie ne peut pas se résumer à un vote tous les cinq ans. Le peuple doit pouvoir révoquer ses élus.",
        "a": "L'article 49.3 a été utilisé 100 fois cette année."
    },
    {
        "p": "Gérald Darmanin", "q": "",
        "ctx": "La lutte contre les stupéfiants est notre combat quotidien. Les opérations 'place nette' déstabilisent les réseaux. Nous saisissons des quantités records de cocaïne et de cannabis. Les points de deal reculent dans nos quartiers populaires. Nous protégeons les plus jeunes de cette économie souterraine. Les moyens de la gendarmerie sont renforcés. Nous créons 200 nouvelles brigades sur tout le territoire. L'ordre est la condition du progrès social.",
        "a": "Il y a plus de policiers à Paris qu'à New York."
    },
    {
        "p": "Yannick Jadot", "q": "",
        "ctx": "Le climat ne nous attend pas. Les rapports du GIEC sont de plus en plus alarmants. Nous devons sortir des énergies fossiles le plus vite possible. Les transports représentent un tiers de nos émissions. Le train doit devenir moins cher que l'avion. Nous devons isoler tous les bâtiments de ce pays. C'est bon pour la planète et pour le porte-monnaie. L'écologie est une opportunité industrielle majeure pour la France. La nature n'est pas une ressource infinie.",
        "a": "La France est le pays le plus polluant au monde par habitant."
    },
    {
        "p": "François Ruffin", "q": "",
        "ctx": "Le travail doit payer. Aujourd'hui, on peut travailler à temps plein et vivre sous le seuil de pauvreté. C'est indigne. La précarité explose dans les métiers du soin et du lien. Les aides à domicile sont les oubliées de la République. Elles font des kilomètres sans être remboursées. Il faut un salaire minimum européen. Les profits records ne ruissellent pas vers ceux qui produisent. La dignité ouvrière doit être remise au centre.",
        "a": "Le nombre de milliardaires en France a baissé de 50% sous ce mandat."
    }
]

In [1]:

import json
import asyncio

async def run_benchmark_tv_production():
    print(f"\n⏱️  Lancement du test d'affichage OBS ({len(dataset_pipeline_expert)} tests)...")
    print("="*80)

    for i, item in enumerate(dataset_pipeline_expert):
        try:
            start_time = time.perf_counter()
            
            # Préparation des données pour ce test
            data_pour_agents = {
                "personne": item['p'],
                "question_posee": item.get('q', ''),
                "affirmation": item['a'],
                "contexte_precedent": item.get('ctx', '')
            }

            print(f"\n🎬 TEST {i+1}/{len(dataset_pipeline_expert)} | Sujet : {item['p']}")
            print(f"🗣️  PHRASE : '{item['a']}'")
            
            # --- 1. ANALYSE PARALLÈLE ---
            # Le routeur est appelé à l'intérieur, et la fonction renvoie directement les rapports
            rapports_bruts = await executer_analyse_parallele(data_pour_agents)
            
            # --- 2. ARBITRAGE DU JUGE ---
            juge_output = await agent_editeur(
                contexte_precedent=data_pour_agents["contexte_precedent"], 
                affirmation_actuelle=data_pour_agents["affirmation"], 
                rapports_agents=rapports_bruts
            )
            
            # --- 3. ENRICHISSEMENT DES SOURCES ---
            # C'est ici qu'on colle les URL trouvées par l'Agent Stat dans le JSON du Juge
            if isinstance(juge_output, dict):
                bandeau_tv_final = _enrich_editor_result_with_sources(juge_output, rapports_bruts)
            else:
                bandeau_tv_final = {"afficher_bandeau": False, "raison": "Erreur format Juge."}

            # --- 4. LOGIQUE D'AFFICHAGE OBS ---
            afficher = bandeau_tv_final.get('afficher_bandeau', False)
            verdict = bandeau_tv_final.get('verdict_global', 'Indéterminé')
            
            if afficher:
                # On parcourt les explications (stats, contexte, etc.) pour trouver le texte à afficher
                explications = bandeau_tv_final.get('explications', {})
                for agent_nom, data_exp in explications.items():
                    if isinstance(data_exp, dict) and "texte" in data_exp:
                        texte_obs = data_exp.get("texte", "")
                        source_nom = data_exp.get("source", "Source officielle")
                        # 🚨 C'est ce print qui simule l'envoi à ta régie vidéo
                        print(f"🚨 [ENVOI OBS] -> {texte_obs} (Source : {source_nom})")
            else:
                raison = bandeau_tv_final.get('raison', '')
                print(f"🔇 [SILENCE OBS] (Verdict: {verdict}) {raison}")

            end_time = time.perf_counter()
            print(f"⏱️  Temps : {end_time - start_time:.2f}s")
            print("-" * 80)

        except Exception as e:
            import traceback
            print(f"❌ Erreur critique au test {i+1} : {e}")
            traceback.print_exc()
        
        # Pause pour éviter de déclencher le Rate Limit de Mistral
        await asyncio.sleep(2)

# Lancement de la boucle
await run_benchmark_tv_production()

await run_benchmark_tv_production()

NameError: name 'dataset_pipeline_expert' is not defined

In [ ]:
pip install "pandas<3.0.0" "numpy<2.4.0" --force-reinstall
